In [ ]:

import pandas as pd
import math

# Step 1: Start

# Step 2: Create training dataset
data = pd.DataFrame({
    'Outlook': [
        'Sunny', 'Sunny', 'Overcast', 'Rain',
        'Rain', 'Rain', 'Overcast', 'Sunny',
        'Sunny', 'Rain', 'Sunny', 'Overcast',
        'Overcast', 'Rain'
    ],

    'Temperature': [
        'Hot', 'Hot', 'Hot', 'Mild',
        'Cool', 'Cool', 'Cool', 'Mild',
        'Cool', 'Mild', 'Mild', 'Mild',
        'Hot', 'Mild'
    ],

    'Humidity': [
        'High', 'High', 'High', 'High',
        'Normal', 'Normal', 'Normal', 'High',
        'Normal', 'Normal', 'Normal', 'High',
        'Normal', 'High'
    ],

    'Wind': [
        'Weak', 'Strong', 'Weak', 'Weak',
        'Weak', 'Strong', 'Strong', 'Weak',
        'Weak', 'Weak', 'Strong', 'Strong',
        'Weak', 'Strong'
    ],

    'Play': [
        'No', 'No', 'Yes', 'Yes',
        'Yes', 'No', 'Yes', 'No',
        'Yes', 'Yes', 'Yes', 'Yes',
        'Yes', 'No'
    ]
})

target = 'Play'

print("Training Dataset:")
print(data)


# Step 3: Calculate class probabilities

def class_probabilities(y):
    return y.value_counts(normalize=True)


# Step 4: Calculate Gini and Entropy

def gini(y):
    p = class_probabilities(y)
    return 1 - sum(p ** 2)


def entropy(y):
    p = class_probabilities(y)
    return -sum(x * math.log2(x) for x in p if x > 0)


def get_splits(data, feature):
    return data[feature].unique()


def weighted_gini(left, right):
    total = len(left) + len(right)

    return (
        len(left) / total * gini(left[target])
        + len(right) / total * gini(right[target])
    )


def information_gain(parent, left, right):
    total = len(parent)

    return (
        entropy(parent[target])
        - (len(left) / total * entropy(left[target])
        + len(right) / total * entropy(right[target]))
    )


def best_split(data, criterion):

    best_feature = None
    best_value = None

    if criterion == 'gini':
        best_score = float('inf')
    else:
        best_score = -float('inf')

    features = [c for c in data.columns if c != target]

    for feature in features:

        for value in get_splits(data, feature):

            left = data[data[feature] == value]
            right = data[data[feature] != value]

            if len(left) == 0 or len(right) == 0:
                continue

            if criterion == 'gini':

                score = weighted_gini(left, right)

                if score < best_score:
                    best_score = score
                    best_feature = feature
                    best_value = value

            else:

                score = information_gain(data, left, right)

                if score > best_score:
                    best_score = score
                    best_feature = feature
                    best_value = value

    return best_feature, best_value, best_score


def build_tree(data, depth=0, max_depth=5, criterion='gini'):

    # Pure node
    if len(data[target].unique()) == 1:
        return data[target].iloc[0]

    # Maximum depth
    if depth >= max_depth:
        return data[target].mode()[0]

    feature, value, score = best_split(data, criterion)

    if feature is None:
        return data[target].mode()[0]

    if criterion == 'gini':
        if score >= gini(data[target]):
            return data[target].mode()[0]
    else:
        if score <= 0:
            return data[target].mode()[0]

    left = data[data[feature] == value]
    right = data[data[feature] != value]

    return {
        'feature': feature,
        'value': value,
        'left': build_tree(left, depth + 1, max_depth, criterion),
        'right': build_tree(right, depth + 1, max_depth, criterion)
    }


criterion = 'gini'

tree = build_tree(data, criterion=criterion)

print("\nDecision Tree:")
print(tree)

def predict(row, tree):

    if not isinstance(tree, dict):
        return tree

    feature = tree['feature']
    value = tree['value']

    if row[feature] == value:
        return predict(row, tree['left'])
    else:
        return predict(row, tree['right'])

new_input = {
    'Outlook': 'Sunny',
    'Temperature': 'Cool',
    'Humidity': 'High',
    'Wind': 'Strong'
}

result = predict(new_input, tree)

print("\nNew Input:", new_input)
print("Predicted Class:", result)

print("\nDecision Tree Classification Completed!")


Training Dataset:
     Outlook Temperature Humidity    Wind Play
0      Sunny         Hot     High    Weak   No
1      Sunny         Hot     High  Strong   No
2   Overcast         Hot     High    Weak  Yes
3       Rain        Mild     High    Weak  Yes
4       Rain        Cool   Normal    Weak  Yes
5       Rain        Cool   Normal  Strong   No
6   Overcast        Cool   Normal  Strong  Yes
7      Sunny        Mild     High    Weak   No
8      Sunny        Cool   Normal    Weak  Yes
9       Rain        Mild   Normal    Weak  Yes
10     Sunny        Mild   Normal  Strong  Yes
11  Overcast        Mild     High  Strong  Yes
12  Overcast         Hot   Normal    Weak  Yes
13      Rain        Mild     High  Strong   No

Decision Tree:
{'feature': 'Outlook', 'value': 'Overcast', 'left': 'Yes', 'right': {'feature': 'Humidity', 'value': 'High', 'left': {'feature': 'Outlook', 'value': 'Sunny', 'left': 'No', 'right': {'feature': 'Wind', 'value': 'Weak', 'left': 'Yes', 'right': 'No'}}, 'right': {'

In [ ]:

# LOGISTIC REGRESSION FOR BINARY CLASSIFICATION

import numpy as np
import pandas as pd

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report
)


# Step 2: Collect the dataset

dataset = load_breast_cancer()

X = pd.DataFrame(
    dataset.data,
    columns=dataset.feature_names
)

y = dataset.target

print("Dataset:")
print(X.head())

print("\nTarget classes:")
print("0 = Malignant")
print("1 = Benign")


# Step 3: Split dataset into training and testing data

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)


# Step 4: Standardize the input features

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)

X_test = scaler.transform(X_test)


# Step 5: Create Logistic Regression model

model = LogisticRegression()

model.fit(X_train, y_train)

probabilities = model.predict_proba(X_test)

print("\nPredicted Probabilities:")
print(probabilities[:5])

y_pred = model.predict(X_test)

print("\nPredicted Classes:")
print(y_pred[:10])

accuracy = accuracy_score(y_test, y_pred)

print("\nAccuracy:", accuracy * 100, "%")


print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))


print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred,
    target_names=dataset.target_names
))

new_input = X.iloc[[0]]

new_input_scaled = scaler.transform(new_input)

prediction = model.predict(new_input_scaled)

probability = model.predict_proba(new_input_scaled)

print("\nNew Input Prediction:")

if prediction[0] == 0:
    print("Predicted Class: Malignant")
else:
    print("Predicted Class: Benign")

print("Probability:", probability[0])

print("\nLogistic Regression Completed!")


Dataset:
   mean radius  mean texture  mean perimeter  mean area  mean smoothness  \
0        17.99         10.38          122.80     1001.0          0.11840   
1        20.57         17.77          132.90     1326.0          0.08474   
2        19.69         21.25          130.00     1203.0          0.10960   
3        11.42         20.38           77.58      386.1          0.14250   
4        20.29         14.34          135.10     1297.0          0.10030   

   mean compactness  mean concavity  mean concave points  mean symmetry  \
0           0.27760          0.3001              0.14710         0.2419   
1           0.07864          0.0869              0.07017         0.1812   
2           0.15990          0.1974              0.12790         0.2069   
3           0.28390          0.2414              0.10520         0.2597   
4           0.13280          0.1980              0.10430         0.1809   

   mean fractal dimension  ...  worst radius  worst texture  worst perimeter  \
0  